# SAM 3 fish segmentation

Loads a SAM 3 image model once, then applies the text prompt `fish` to every image in an input directory. Masks and transparent overlays are saved to an output directory.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw

from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

In [ ]:
# Paths and segmentation settings
INPUT_DIR = Path(r"C:\Users\jakob\Pictures\ALL_IMAGES_RULER\Measured")
OUTPUT_DIR = Path(r"C:\Users\jakob\Pictures\SEGMENTED_FISH")
PROMPT = "fish"
MIN_SCORE = 0.5
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Input:  {INPUT_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
def load_sam3():
    """Load SAM 3 once and prepare it for text-prompted image segmentation."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_sam3_image_model()
    model = model.to(device)
    model.eval()
    processor = Sam3Processor(model)
    print(f"SAM 3 loaded on {device}")
    return model, processor, device


model, processor, device = load_sam3()

In [ ]:
def segment_image(image, prompt=PROMPT, min_score=MIN_SCORE):
    """Return filtered boolean masks and scores for one PIL RGB image."""
    with torch.inference_mode():
        state = processor.set_image(image)
        result = processor.set_text_prompt(state=state, prompt=prompt)

    masks = result["masks"]
    if hasattr(masks, "detach"):
        masks = masks.detach().cpu()
    masks = torch.as_tensor(masks)

    if masks.ndim == 2:
        masks = masks.unsqueeze(0)
    elif masks.ndim == 4:
        masks = masks.squeeze(1)

    masks = masks.numpy().astype(bool)
    scores = result["scores"]
    if hasattr(scores, "detach"):
        scores = scores.detach().cpu()
    scores = torch.as_tensor(scores).reshape(-1).numpy()

    keep = scores >= min_score
    return masks[keep], scores[keep]


def save_segmentation(image, stem, masks, scores, output_dir):
    """Save an instance mask, a combined mask, and a transparent RGB overlay."""
    output_dir.mkdir(parents=True, exist_ok=True)
    width, height = image.size
    combined = np.zeros((height, width), dtype=bool)

    rgba = np.array(image.convert("RGBA"), dtype=np.uint8)

    for index, (mask, score) in enumerate(zip(masks, scores), start=1):
        mask = np.asarray(mask, dtype=bool)
        combined |= mask
        Image.fromarray(mask.astype(np.uint8) * 255).save(
            output_dir / f"{stem}_{PROMPT}_{index:03d}_mask.png"
        )

        # Stable semi-transparent green overlay for each instance.
        rng = np.random.default_rng(index)
        color = rng.integers(40, 256, size=3, dtype=np.uint8)
        rgba[mask, :3] = (
            rgba[mask, :3].astype(np.uint16) * 55
            + color.astype(np.uint16) * 45
        ) // 100

    Image.fromarray(combined.astype(np.uint8) * 255).save(
        output_dir / f"{stem}_{PROMPT}_combined_mask.png"
    )
    Image.fromarray(rgba, mode="RGBA").save(
        output_dir / f"{stem}_{PROMPT}_overlay.png"
    )

    with open(output_dir / f"{stem}_{PROMPT}_scores.csv", "w", newline="") as file:
        file.write("instance,score\n")
        for index, score in enumerate(scores, start=1):
            file.write(f"{index},{float(score):.6f}\n")

    return combined

In [ ]:
def segment_directory(input_dir=INPUT_DIR, output_dir=OUTPUT_DIR,
                      prompt=PROMPT, min_score=MIN_SCORE):
    """Segment supported images in input_dir and save masks and overlays."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory does not exist: {input_dir}")

    paths = sorted(
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
    )
    if not paths:
        raise ValueError(f"No supported images found in {input_dir}")

    results = {}
    csv_path = output_dir / "sam3_segmentation_summary.csv"

    with open(csv_path, "w", newline="") as file:
        file.write("image,instances,max_score\n")

        for path in paths:
            image = Image.open(path).convert("RGB")
            masks, scores = segment_image(image, prompt=prompt, min_score=min_score)
            combined = save_segmentation(
                image, path.stem, masks, scores, output_dir
            )

            max_score = float(scores.max()) if len(scores) else 0.0
            file.write(f"{path.name},{len(masks)},{max_score:.6f}\n")
            results[path.name] = {
                "masks": masks,
                "scores": scores,
                "combined_mask": combined,
            }
            print(f"{path.name}: {len(masks)} instance(s)")

    print(f"Processed {len(results)} image(s)")
    print(f"Summary: {csv_path}")
    return results

In [ ]:
results = segment_directory(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    prompt=PROMPT,
    min_score=MIN_SCORE,
)